<a href="https://colab.research.google.com/github/dbellavista-ai/internship-deepfake-forensic/blob/main/step2_retrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 - Project Setup
This section initializes the complete workspace environment, mounts Google Drive and extracts the dataset from the .zip archive directly to the local instance storage (/content/dataset).


In [ ]:
!pip install -q timm

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Extract the dataset ONLY if it hasn't been extracted in this session
dataset_dest_path = '/content/dataset'

if not os.path.exists(dataset_dest_path):
    print("Extracting the dataset... (this might take a few minutes)")
    !unzip -q /content/drive/MyDrive/internship-deepfake-forensic/deepfake_dataset.zip -d {dataset_dest_path}
    print("Extraction completed!")
else:
    print("Dataset already present and ready to use!")

## 2 - Dataset and Advanced Augmentation

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_dirs, fake_dirs, transform=None):
        self.filepaths, self.labels = [], []
        self.transform = transform

        for d in real_dirs:
            for ext in ('*.png', '*.jpg', '*.jpeg'):
                paths = glob.glob(os.path.join(d, ext))
                self.filepaths.extend(paths)
                self.labels.extend([0] * len(paths))

        for d in fake_dirs:
            for ext in ('*.png', '*.jpg', '*.jpeg'):
                paths = glob.glob(os.path.join(d, ext))
                self.filepaths.extend(paths)
                self.labels.extend([1] * len(paths))

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor([self.labels[idx]], dtype=torch.float32)
        return img, label

# ADVANCED AUGMENTATION
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0)
])

val_test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

## 3 - Models Architecture

In [ ]:
class StandardEfficientNet(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

class DeitTiny(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.model = timm.create_model('deit_tiny_patch16_224', pretrained=True)
        n_features = self.model.head.in_features
        self.model.head = nn.Linear(n_features, num_classes)

    def forward(self, x):
        return self.model(x)

class HybridEfficientNet(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).features
        in_channels = 1280
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1, bias=False),
            nn.BatchNorm2d(in_channels // 8), nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, in_channels, 1, bias=False), nn.Sigmoid()
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(in_channels, 256), nn.ReLU(inplace=True),
            nn.Dropout(p=0.4), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        features = features * self.attention(features)
        return self.classifier(torch.flatten(self.pool(features), 1))

## 4 - Training Engine

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001, path='best_model.pth'):
        self.patience, self.delta, self.path = patience, delta, path
        self.counter, self.best_score, self.early_stop = 0, None, False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    #ModelCheckPoint - save the model when improve
    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def train_and_validate(model, train_loader, val_loader, epochs, optimizer, criterion, device, early_stopper, scheduler=None):
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(epochs):
        # TRAINING PHASE
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="batch", leave=False)

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)

            predicted = (outputs > 0.0).float()
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            train_bar.set_postfix(loss=loss.item(), acc=train_correct/train_total)

        # VALIDATION PHASE
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", unit="batch", leave=False)

        with torch.no_grad():
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                predicted = (outputs > 0.0).float()
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        avg_train_loss = train_loss / train_total
        avg_val_loss = val_loss / val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total

        print(f"END EPOCH {epoch+1} -> Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.4f}")

        # SCHEDULER LOGIC
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_val_loss)
            else:
                scheduler.step()

        early_stopper(avg_val_loss, model)
        if early_stopper.early_stop:
            print(">> Early stopping activate!")
            break
        print("-" * 30)

def load_or_train(model, model_path, train_loader, val_loader, epochs, optimizer, criterion, device, early_stopper, scheduler=None):
    if os.path.exists(model_path):
        print(f"Model found! Skipping training and loading weights from:\n{model_path}")
        model.load_state_dict(torch.load(model_path, map_location=device))
    else:
        print(f"Model not found. Starting the training procedure...")
        train_and_validate(
            model, train_loader, val_loader, epochs, optimizer,
            criterion, device, early_stopper, scheduler
        )
        # Load the best model saved by EarlyStopping at the end of training
        model.load_state_dict(torch.load(model_path, map_location=device))

## 5 - Data Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

path_intra = '/content/drive/MyDrive/internship-deepfake-forensic/deepfake_models/retrain'
os.makedirs(path_intra, exist_ok=True)
print(f"In this second step the models are saved in: {path_intra}")

BATCH_SIZE = 32
MAX_EPOCHS = 15

# --- SETUP P1 (Source-Based) ---
tr_reals_p1 = ['/content/dataset/FEI_Split_Intra/train/original/', '/content/dataset/F++_Split_Intra/source_based/train/original/']
tr_fakes_p1 = ['/content/dataset/FEI_Split_Intra/train/fake/', '/content/dataset/F++_Split_Intra/source_based/train/fake/']
vl_reals_p1 = ['/content/dataset/FEI_Split_Intra/val/original/', '/content/dataset/F++_Split_Intra/source_based/val/original/']
vl_fakes_p1 = ['/content/dataset/FEI_Split_Intra/val/fake/', '/content/dataset/F++_Split_Intra/source_based/val/fake/']

# --- SETUP P2 (Target-Based) ---
tr_reals_p2 = ['/content/dataset/FEI_Split_Intra/train/original/', '/content/dataset/F++_Split_Intra/target_based/train/original/']
tr_fakes_p2 = ['/content/dataset/FEI_Split_Intra/train/fake/', '/content/dataset/F++_Split_Intra/target_based/train/fake/']
vl_reals_p2 = ['/content/dataset/FEI_Split_Intra/val/original/', '/content/dataset/F++_Split_Intra/target_based/val/original/']
vl_fakes_p2 = ['/content/dataset/FEI_Split_Intra/val/fake/', '/content/dataset/F++_Split_Intra/target_based/val/fake/']

def get_criterion_for_split(reals_dirs, fakes_dirs):
    num_reals = sum([len(glob.glob(os.path.join(d, '*.*'))) for d in reals_dirs])
    num_fakes = sum([len(glob.glob(os.path.join(d, '*.*'))) for d in fakes_dirs])
    balance_ratio = num_reals / num_fakes if num_fakes > 0 else 1.0
    print(f"Class balance computed: {num_reals} Real / {num_fakes} Fake (pos_weight: {balance_ratio:.4f})")
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([balance_ratio]).to(device))

print("Computing class weights for P1:")
criterion_p1 = get_criterion_for_split(tr_reals_p1, tr_fakes_p1)
print("\nComputing class weights for P2:")
criterion_p2 = get_criterion_for_split(tr_reals_p2, tr_fakes_p2)

# DataLoaders
train_loader_p1 = DataLoader(DeepfakeDataset(tr_reals_p1, tr_fakes_p1, train_transforms), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader_p1   = DataLoader(DeepfakeDataset(vl_reals_p1, vl_fakes_p1, val_test_transforms), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

train_loader_p2 = DataLoader(DeepfakeDataset(tr_reals_p2, tr_fakes_p2, train_transforms), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader_p2   = DataLoader(DeepfakeDataset(vl_reals_p2, vl_fakes_p2, val_test_transforms), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

## 6 - Training P1 and P2

### 6.1 - EfficientNet-B0: Phase 1

In [ ]:
path_baseline = '/content/drive/MyDrive/internship-deepfake-forensic/deepfake_models/baseline'

print("\n--- STANDARD EFFICIENTNET-B0 --> START PHASE 1 (FINE-TUNING): FEI + F++ SOURCE BASED ---")

model_std_b0_p1 = StandardEfficientNet().to(device)

# 2. LOAD BASELINE WEIGHTS (If they exist)
baseline_std_b0_path_p1 = os.path.join(path_baseline, "std_b0_P1_source_intra_step1.pth")
if os.path.exists(baseline_std_b0_path_p1):
    print(f"Loading Baseline weights from: {baseline_std_b0_path_p1}")
    model_std_b0_p1.load_state_dict(torch.load(baseline_std_b0_path_p1, map_location=device))
else:
    print("WARNING: Baseline file not found! Training will start from scratch.")

optimizer_b0_p1 = optim.AdamW(model_std_b0_p1.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler_std_b0_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b0_p1, mode='min', factor=0.5, patience=2
)
path_std_b0_p1 = os.path.join(path_intra, "std_b0_P1_source_intra_step2.pth")
early_stopper_std_b0_p1 = EarlyStopping(patience=5, path=path_std_b0_p1)

load_or_train(
    model_std_b0_p1,
    path_std_b0_p1,
    train_loader_p1,
    val_loader_p1,
    MAX_EPOCHS,
    optimizer_b0_p1,
    criterion_p1,
    device,
    early_stopper_std_b0_p1,
    scheduler_std_b0_p1
)

### 6.2 - EfficientNet-B0: Phase 2

In [ ]:
print("\n--- STANDARD EFFICIENTNET-B0 --> START PHASE 2 (FINE-TUNING): FEI + F++ TARGET BASED ---")

model_std_b0_p2 = StandardEfficientNet().to(device)

# LOAD BASELINE WEIGHTS (If they exist)
baseline_std_b0_path_p2 = os.path.join(path_baseline, "std_b0_P2_target_intra_step1.pth")
if os.path.exists(baseline_std_b0_path_p2):
    print(f"Loading Baseline weights from: {baseline_std_b0_path_p2}")
    model_std_b0_p2.load_state_dict(torch.load(baseline_std_b0_path_p2, map_location=device))
else:
    print("WARNING: Baseline file not found! Training will start from scratch.")

optimizer_b0_p2 = optim.AdamW(model_std_b0_p2.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler_std_b0_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b0_p2, mode='min', factor=0.5, patience=2
)
path_std_b0_p2 = os.path.join(path_intra, "std_b0_P2_target_intra_step2.pth")
early_stopper_std_b0_p2 = EarlyStopping(patience=5, path=path_std_b0_p2)

load_or_train(
    model_std_b0_p2,
    path_std_b0_p2,
    train_loader_p2,
    val_loader_p2,
    MAX_EPOCHS,
    optimizer_b0_p2,
    criterion_p2,
    device,
    early_stopper_std_b0_p2,
    scheduler_std_b0_p2
)

### 6.3 - Deit-Tiny: Phase 1

In [ ]:
print("\n--- DEIT-TINY --> START PHASE 1 (FINE-TUNING): FEI + F++ SOURCE BASED ---")

model_deit_p1 = DeitTiny().to(device)

# LOAD BASELINE WEIGHTS (If they exist)
baseline_deit_path_p1 = os.path.join(path_baseline, "deit_P1_source_intra_step1.pth")
if os.path.exists(baseline_deit_path_p1):
    print(f"Loading Baseline weights from: {baseline_deit_path_p1}")
    model_deit_p1.load_state_dict(torch.load(baseline_deit_path_p1, map_location=device))
else:
    print("WARNING: Baseline file not found! Training will start from scratch.")

optimizer_deit_p1 = optim.AdamW(model_deit_p1.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler_deit_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_deit_p1, mode='min', factor=0.5, patience=2
)
path_deit_p1 = os.path.join(path_intra, "deit_P1_source_intra_step2.pth")
early_stopper_deit_p1 = EarlyStopping(patience=5, path=path_deit_p1)

load_or_train(
    model_deit_p1,
    path_deit_p1,
    train_loader_p1,
    val_loader_p1,
    MAX_EPOCHS,
    optimizer_deit_p1,
    criterion_p1,
    device,
    early_stopper_deit_p1,
    scheduler_deit_p1
)

### 6.4 - Deit-Tiny: Phase 2

In [ ]:
print("\n--- DEIT-TINY --> START PHASE 2 (FINE-TUNING): FEI + F++ TARGET BASED ---")

model_deit_p2 = DeitTiny().to(device)

# LOAD BASELINE WEIGHTS (If they exist)
baseline_deit_path_p2 = os.path.join(path_baseline, "deit_P2_target_intra_step1.pth")
if os.path.exists(baseline_deit_path_p2):
    print(f"Loading Baseline weights from: {baseline_deit_path_p2}")
    model_deit_p2.load_state_dict(torch.load(baseline_deit_path_p2, map_location=device))
else:
    print("WARNING: Baseline file not found! Training will start from scratch.")

optimizer_deit_p2 = optim.AdamW(model_deit_p2.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler_deit_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_deit_p2, mode='min', factor=0.5, patience=2
)
path_deit_p2 = os.path.join(path_intra, "deit_P2_target_intra_step2.pth")
early_stopper_deit_p2 = EarlyStopping(patience=5, path=path_deit_p2)

load_or_train(
    model_deit_p2,
    path_deit_p2,
    train_loader_p2,
    val_loader_p2,
    MAX_EPOCHS,
    optimizer_deit_p2,
    criterion_p2,
    device,
    early_stopper_deit_p2,
    scheduler_deit_p2
)

### 6.5 - Hybrid EfficientNet-B0: Phase 1

In [ ]:
print("\n--- HYBRID EFFICIENTNET-B0 --> START PHASE 1 (FINE-TUNING): FEI + F++ SOURCE BASED ---")

model_hyb_p1 = HybridEfficientNet().to(device)

# LOAD BASELINE WEIGHTS (If they exist)
baseline_hyb_path_p1 = os.path.join(path_baseline, "hyb_b0_P1_source_intra_step1.pth")
if os.path.exists(baseline_hyb_path_p1):
    print(f"Loading Baseline weights from: {baseline_hyb_path_p1}")
    model_hyb_p1.load_state_dict(torch.load(baseline_hyb_path_p1, map_location=device))
else:
    print("WARNING: Baseline file not found! Training will start from scratch.")

optimizer_hyb_p1 = optim.AdamW(model_hyb_p1.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler_hyb_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_hyb_p1, mode='min', factor=0.5, patience=2
)
path_hyb_p1 = os.path.join(path_intra, "hyb_b0_P1_source_intra_step2.pth")
early_stopper_hyb_p1 = EarlyStopping(patience=5, path=path_hyb_p1)

load_or_train(
    model_hyb_p1,
    path_hyb_p1,
    train_loader_p1,
    val_loader_p1,
    MAX_EPOCHS,
    optimizer_hyb_p1,
    criterion_p1,
    device,
    early_stopper_hyb_p1,
    scheduler_hyb_p1
)

### 6.6 - Hybrid EfficientNet-B0: Phase 2

In [ ]:
print("\n--- HYBRID EFFICIENTNET-B0 --> START PHASE 2 (FINE-TUNING): FEI + F++ TARGET BASED ---")

model_hyb_p2 = HybridEfficientNet().to(device)

# LOAD BASELINE WEIGHTS (If they exist)
baseline_hyb_path_p2 = os.path.join(path_baseline, "hyb_b0_P2_target_intra_step1.pth")
if os.path.exists(baseline_hyb_path_p2):
    print(f"Loading Baseline weights from: {baseline_hyb_path_p2}")
    model_hyb_p2.load_state_dict(torch.load(baseline_hyb_path_p2, map_location=device))
else:
    print("WARNING: Baseline file not found! Training will start from scratch.")

optimizer_hyb_p2 = optim.AdamW(model_hyb_p2.parameters(), lr=1e-5, weight_decay=1e-5)
scheduler_hyb_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_hyb_p2, mode='min', factor=0.5, patience=2
)
path_hyb_p2 = os.path.join(path_intra, "hyb_b0_P2_target_intra_step2.pth")
early_stopper_hyb_p2 = EarlyStopping(patience=5, path=path_hyb_p2)

load_or_train(
    model_hyb_p2,
    path_hyb_p2,
    train_loader_p2,
    val_loader_p2,
    MAX_EPOCHS,
    optimizer_hyb_p2,
    criterion_p2,
    device,
    early_stopper_hyb_p2,
    scheduler_hyb_p2
)